# Multi-Defect Model Comparison and Uncertainty Analysis

This notebook compares two approaches to modeling radiation damage in 4H-SiC detectors:

1. **Three-defect model** (Burin 2024): Explicitly models Z1/2, EH6/7, and EH4 defects with
   individual introduction rates and capture cross-sections.
2. **Single-effective-defect model**: Lumps all three defects into one effective defect with
   capture cross-section chosen to match the three-defect model's $K_\tau$.

The single-defect model uses $\eta_{\text{eff}} = \sum_i \eta_i$ and derives
$\sigma_{\text{eff}} = K_\tau / (\eta_{\text{eff}} \cdot v_{\text{th}})$ to ensure identical
lifetime degradation.

We also explore per-defect uncertainty by scaling individual $\eta$ values to produce
CCE uncertainty envelopes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.radiation_damage import (
    RadiationDamageParams, compute_K_tau,
    make_single_defect_params, cce_uncertainty_envelope,
)
from src.charge_collection import cce_vs_fluence
from src.dark_current import dark_current_vs_fluence
from src.cv_analysis import cv_at_fluence

plt.rcParams.update({"font.family": "serif", "font.size": 12,
                      "axes.labelsize": 14, "figure.dpi": 150})

In [ ]:
# Construct both models
three_defect = RadiationDamageParams()
single_defect = make_single_defect_params()

# Compare K_tau for both carriers
for carrier in ["electron", "hole"]:
    K_3 = compute_K_tau(three_defect, carrier)
    K_1 = compute_K_tau(single_defect, carrier)
    print(f"K_tau ({carrier}):")
    print(f"  Three-defect: {K_3:.6e} cm^2/s")
    print(f"  Single-defect: {K_1:.6e} cm^2/s")
    print(f"  Relative error: {abs(K_3 - K_1) / K_3:.2e}")
    print()

In [ ]:
# CCE vs Fluence comparison
fluences = [0] + list(np.geomspace(1e10, 5e13, 10))

cce_3 = cce_vs_fluence(fluences, V_bias=-40, damage_params=three_defect)
cce_1 = cce_vs_fluence(fluences, V_bias=-40, damage_params=single_defect)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(fluences[1:], cce_3[1:], 'o-', label='Three-defect', markersize=6)
ax.semilogx(fluences[1:], cce_1[1:], 's--', label='Single-defect', markersize=6)
ax.set_xlabel(r'Proton fluence (cm$^{-2}$)')
ax.set_ylabel('Charge Collection Efficiency')
ax.set_title('CCE vs Fluence: Single-Defect vs Three-Defect Model')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig('figures/12a_cce_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Dark current comparison
J_dark_3 = dark_current_vs_fluence(fluences, V_bias=-30, damage_params=three_defect)
J_dark_1 = dark_current_vs_fluence(fluences, V_bias=-30, damage_params=single_defect)

fig, ax = plt.subplots(figsize=(8, 5))
# Skip zero fluence for log-log
ax.loglog(fluences[1:], np.abs(J_dark_3[1:]), 'o-', label='Three-defect', markersize=6)
ax.loglog(fluences[1:], np.abs(J_dark_1[1:]), 's--', label='Single-defect', markersize=6)
ax.set_xlabel(r'Proton fluence (cm$^{-2}$)')
ax.set_ylabel(r'Dark current density (A/cm$^2$)')
ax.set_title('Dark Current vs Fluence: Single-Defect vs Three-Defect Model')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('figures/12b_dark_current_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Note: Total dark current is TAT-dominated (N_t term, ~4 orders of")
print("magnitude above SRH), so both models give near-identical results.")

In [ ]:
# C-V comparison at fluence = 1e13
V_range = np.linspace(-100, 0, 200)
cv_3 = cv_at_fluence(1e13, V_range, damage_params=three_defect)
cv_1 = cv_at_fluence(1e13, V_range, damage_params=single_defect)

fig, ax = plt.subplots(figsize=(8, 5))
if cv_3 is not None:
    ax.plot(V_range, cv_3 * 1e12, '-', label='Three-defect', linewidth=2)
if cv_1 is not None:
    ax.plot(V_range, cv_1 * 1e12, '--', label='Single-defect', linewidth=2)
ax.set_xlabel('Bias voltage (V)')
ax.set_ylabel('Capacitance (pF/cm$^2$)')
ax.set_title(r'C-V at $\Phi = 10^{13}$ p/cm$^2$: Single vs Three-Defect')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('figures/12c_cv_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## When Do Single-Defect and Three-Defect Models Differ?

The comparisons above show that the single-defect and three-defect models produce
**equivalent results** for:

- **CCE vs fluence**: Both models yield the same $K_\tau$, so lifetime degradation
  and charge collection are identical.
- **Dark current**: The TAT-dominated dark current depends on total $N_t = \sum \eta_i \cdot \Phi$,
  which is identical since $\eta_{\text{eff}} = \sum \eta_i$.
- **C-V characteristics**: Carrier removal uses $\eta_{\text{removal}}$, which is the same
  for both models.

The key differences emerge in:

1. **Annealing predictions**: Different defects anneal at different rates (Z1/2 has
   $E_a \approx 4.5$ eV, EH6/7 and EH4 have different activation energies). A single-defect
   model cannot capture differential annealing.
2. **Physical interpretation**: The three-defect model provides physical defect concentrations
   that can be compared with DLTS measurements.
3. **Per-defect uncertainty**: When defect introduction rates have different uncertainties,
   the three-defect model enables per-defect sensitivity analysis (shown below).

In [ ]:
# Per-defect uncertainty bands
fluences_unc = np.geomspace(1e10, 5e13, 30)

envelope = cce_uncertainty_envelope(fluences_unc, V_bias=-40,
                                     scale_low=0.5, scale_high=2.0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(fluences_unc, envelope['cce_nominal'], 'k-', linewidth=2, label='Nominal')
ax.fill_between(fluences_unc, envelope['cce_min'], envelope['cce_max'],
                alpha=0.3, color='steelblue', label=r'0.5$\times$-2.0$\times$ $\eta$ scatter')
ax.set_xlabel(r'Proton fluence (cm$^{-2}$)')
ax.set_ylabel('Charge Collection Efficiency')
ax.set_title(r'CCE vs Fluence with Per-Defect Uncertainty (0.5$\times$-2.0$\times$ $\eta$ scatter)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig('figures/12d_cce_uncertainty_bands.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Multi-bias uncertainty envelopes
V_biases = [-20, -40, -60, -100]
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

fig, ax = plt.subplots(figsize=(9, 6))
for V_bias, color in zip(V_biases, colors):
    env = cce_uncertainty_envelope(fluences_unc, V_bias=V_bias,
                                    scale_low=0.5, scale_high=2.0)
    ax.semilogx(fluences_unc, env['cce_nominal'], '-', color=color,
                linewidth=2, label=f'V = {V_bias} V')
    ax.fill_between(fluences_unc, env['cce_min'], env['cce_max'],
                    alpha=0.15, color=color)

ax.set_xlabel(r'Proton fluence (cm$^{-2}$)')
ax.set_ylabel('Charge Collection Efficiency')
ax.set_title('CCE Uncertainty Bands at Multiple Bias Voltages')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig('figures/12e_cce_uncertainty_multibias.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table
fluence_test = 1e13
cce_3_test = cce_vs_fluence([fluence_test], V_bias=-40, damage_params=three_defect)[0]
cce_1_test = cce_vs_fluence([fluence_test], V_bias=-40, damage_params=single_defect)[0]

K_tau_n_3 = compute_K_tau(three_defect, 'electron')
K_tau_p_3 = compute_K_tau(three_defect, 'hole')
K_tau_n_1 = compute_K_tau(single_defect, 'electron')
K_tau_p_1 = compute_K_tau(single_defect, 'hole')

print("| Property | Three-Defect | Single-Defect |")
print("|---|---|---|")
print(f"| K_tau_n (cm^2/s) | {K_tau_n_3:.4e} | {K_tau_n_1:.4e} |")
print(f"| K_tau_p (cm^2/s) | {K_tau_p_3:.4e} | {K_tau_p_1:.4e} |")
print(f"| eta_removal (cm^-1) | {three_defect.eta_removal:.2f} | {single_defect.eta_removal:.2f} |")
print(f"| CCE at 1e13 p/cm^2 | {cce_3_test:.4f} | {cce_1_test:.4f} |")